In [ ]:
pip install -q transformers peft trl datasets accelerate 

In [8]:
pip install -U "torchao>=0.16.0"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 55.4 MB/s  0:00:00
  Attempting uninstall: torchao
    Found existing installation: torchao 0.10.0
    Uninstalling torchao-0.10.0:
      Successfully uninstalled torchao-0.10.0


In [3]:
import torch
# torch = PyTorch library. Tensors, GPU आणि dtype (float16 etc.) साठी लागते.

from datasets import load_dataset
# load_dataset = HuggingFace Hub वरून dataset download/load करण्याचे function.

from transformers import AutoModelForCausalLM, AutoTokenizer
# AutoModelForCausalLM = next-word prediction करणारा LLM (GPT/Llama सारखा) load करतो.
# AutoTokenizer        = text -> token IDs (numbers) आणि परत, convert करतो.

from peft import LoraConfig, get_peft_model
# peft = Parameter-Efficient Fine-Tuning library.
# LoraConfig     = LoRA चे settings (rank, alpha, इ.) ठेवणारा class.
# get_peft_model = normal model ला LoRA adapters लावून "PEFT model" बनवतो.

from trl import SFTTrainer, SFTConfig
# trl = Transformer Reinforcement Learning library; यात SFT (Supervised Fine-Tuning) trainer आहे.
# SFTTrainer = text dataset वर fine-tune करण्याचे ready-made training loop.
# SFTConfig  = training settings (epochs, batch size, lr...) चा class.

# Config

In [4]:
MODEL_NAME = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
# कोणता base model fine-tune करायचा. TinyLlama छोटा (1.1B) आहे, म्हणून free Colab T4 GPU वर चालतो.

DATASET_NAME = "mlabonne/guanaco-llama2-1k"
# 1000 instruction-response examples असलेला छोटा dataset. यात "text" नावाचा column आहे.

OUTPUT_DIR = "./tinyllama-lora"
# training नंतर adapter आणि checkpoints इथे save होतील.

# TOKENIZER 

In [ ]:

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
# Model सोबत train झालेला tokenizer load करतो. Model आणि tokenizer चा pair एकच असला पाहिजे.

tokenizer.pad_token = tokenizer.eos_token
# Batch मधले sentences वेगवेगळ्या लांबीचे असतात, म्हणून "padding" token लागतो.
# TinyLlama ला pad token नसतो, म्हणून end-of-sentence token च pad म्हणून वापरतो.

tokenizer.padding_side = "right"
# Padding उजवीकडे (शेवटी) add कर. Training मध्ये हे stable असते.

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/608 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.29k [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/551 [00:00<?, ?B/s]

# BASE MODEL Download

In [6]:

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    # कोणता model load करायचा.
    torch_dtype=torch.float16,
    # Weights 16-bit float मध्ये load कर -> 32-bit पेक्षा निम्मी memory लागते.
    device_map="auto",
    # Model आपोआप available GPU वर ठेव (GPU नसेल तर CPU).
)

model.config.use_cache = False
# use_cache = generation मधला speed-up (past key/values store करणे). Training मध्ये याचा उपयोग नाही
# आणि gradient checkpointing सोबत warning येते, म्हणून बंद करतो.

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors: reconstructing file:   0%|          |  0.00B / 2.20GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

# Lora CONFIG

In [ ]:

lora_config = LoraConfig(
    r=16,
    # RANK. Adapter matrices चा छोटा dimension. r जास्त = जास्त capacity पण जास्त memory.
    # Normal values: 8, 16, 32, 64.

    lora_alpha=32,
    # SCALING factor. LoRA update ला (alpha / r) ने multiply करतात. Thumb rule: alpha = 2 * r.

    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    # Model मधल्या कोणत्या layers ला adapter लावायचा ते. हे attention layers चे नाव आहेत
    # (Query, Key, Value, Output projection). Model नुसार नावे बदलतात.
    # target_modules="all-linear",  # सगळ्या linear layers ला adapter; layer नावे लक्षात ठेवायची गरज नाही
    lora_dropout=0.05,
    # Adapter वर 5% dropout -> overfitting कमी होते (random neurons बंद करणे).

    task_type="CAUSAL_LM",
    # Task प्रकार: Causal Language Modeling (पुढचा token predict करणे).
)

model = get_peft_model(model, lora_config)
# Base weights FREEZE होतात आणि निवडलेल्या layers मध्ये LoRA adapters (A, B matrices) add होतात.

model.print_trainable_parameters()
# किती parameters trainable आहेत ते print करतो. साधारण ~0.2% दिसेल. हेच LoRA चे जादू आहे!

trainable params: 4,505,600 || all params: 1,104,553,984 || trainable%: 0.4079


# Dataset & Training arg

In [ ]:

dataset = load_dataset(DATASET_NAME, split="train")
# Dataset चा "train" भाग load कर. प्रत्येक row मध्ये "text" column आहे.

# TRAINING ARGS 
training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    # Checkpoints/output कुठे save करायचे.

    num_train_epochs=1,
    # पूर्ण dataset किती वेळा बघायचा. 1 epoch = एकदा पूर्ण data.

    per_device_train_batch_size=4,
    # प्रत्येक GPU वर एका step मध्ये 4 examples. Memory कमी असेल तर 1 किंवा 2 कर.

    gradient_accumulation_steps=4,
    # 4 steps चे gradients साठवून मग एकदा weights update कर.
    # Effective batch size = 4 x 4 = 16. (मोठ्या batch चा effect, कमी memory मध्ये.)

    learning_rate=2e-4,
    # Learning rate. LoRA साठी 1e-4 ते 3e-4 सामान्य (full fine-tune पेक्षा जास्त चालते).

    lr_scheduler_type="cosine",
    # LR हळूहळू cosine curve ने कमी होतो -> शेवटी stable training.

    logging_steps=10,
    # प्रत्येक 10 steps ला loss print कर.

    # save_strategy="epoch",
    # # प्रत्येक epoch च्या शेवटी checkpoint save कर.

    fp16=True,
    # 16-bit mixed precision training -> fast आणि कमी memory. (Ampere+ GPU असेल तर bf16=True वापर.)

    max_length=512,
    # प्रत्येक example जास्तीत जास्त 512 tokens पर्यंत. जास्त लांब असेल तर कापला जाईल.
    # (जुन्या trl versions मध्ये याचे नाव max_seq_length आहे.)


)

README.md:   0%|          | 0.00/1.02k [00:00<?, ?B/s]

data/train-00000-of-00001-9ad84bb9cf65a4(…): reconstructing file:   0%|          |  0.00B /  967kB            

data/train-00000-of-00001-9ad84bb9cf65a4(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

# TRAINER 

In [ ]:

trainer = SFTTrainer(
    model=model,
    # LoRA लावलेला model.
    train_dataset=dataset,
    # Training data.
    args=training_args,
    # वरचे settings.
    processing_class=tokenizer,
    # Tokenizer द्यायचा. (जुन्या trl मध्ये `tokenizer=tokenizer`.)
)

trainer.train()
# प्रत्यक्ष training सुरू. Forward pass -> loss -> backward pass -> फक्त LoRA weights update.


Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (3142 > 2048). Running this sequence through the model will result in indexing errors


Building labels for train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

/usr/local/lib/python3.13/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


In [ ]:
# SAVE 
trainer.save_model(OUTPUT_DIR)

# INFERENCE TEST 

In [ ]:

prompt = "<s>[INST] Explain what LoRA is in two lines. [/INST]"
# Test prompt. Format dataset च्या format सारखा ठेवला आहे (Llama-2 [INST] style).

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
# Text -> token IDs -> PyTorch tensor -> model ज्या device वर आहे (GPU) तिथे पाठव.

model.config.use_cache = True
# Generation मध्ये cache परत चालू कर -> fast output.

with torch.no_grad():
    # no_grad = gradients calculate करू नकोस; inference साठी memory आणि वेळ वाचतो.
    output_ids = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7)
    # generate = नवीन text तयार करतो.
    # max_new_tokens=100 -> जास्तीत जास्त 100 नवीन tokens.
    # do_sample=True, temperature=0.7 -> थोडा creative/random output (कमी temp = जास्त predictable).

print(tokenizer.decode(output_ids[0], skip_special_tokens=True))
# Token IDs -> परत readable text. skip_special_tokens=True म्हणजे <s>, </s> सारखे special tokens काढून टाक.
